# Mission - Construisez et testez une automatisation de transformation et analyse de données


## Contexte

Cette mission suit un scénario de projet professionnel. 


Vous pouvez suivre les étapes pour vous aider à réaliser vos livrables.

 

Avant de démarrer, nous vous conseillons de :

lire toute la mission et ses documents liés ;
prendre des notes sur ce que vous avez compris ;
consulter les étapes pour vous guider ; 
préparer une liste de questions pour votre première session de mentorat.
Prêt à mener la mission ?
Barres titres

 

Vous avez été récemment embauché comme Data Engineer dans l’entreprise BottleNeck, un marchand de vin prestigieux. 

 

Laurent, votre manager sur cette mission, vous accueille chaleureusement et vous propose de partager un café avec le reste de l’équipe. L’ambiance est bonne, et vous voilà déjà parfaitement intégré dans cette équipe détendue mais professionnelle. Vous rencontrez les responsables de produits : 

Laure, qui s'occupe des clients entreprises et des vins premium, 
Maria, qui s’occupe des particuliers et des vins ordinaires.
 

Après les présentations, effectuées dans une ambiance conviviale, Laurent vous explique le contexte data de l’entreprise : 

”Stéphane, notre data analyst a travaillé sur une première analyse de nos données. Il a présenté ses résultats lors de notre dernière réunion de COPIL et on a eu de bons retours de la part de nos responsables produits. 

Stéphane a travaillé sur ces 3 axes :  

Nettoyage des données provenant des 2 systèmes CMS et ERP.
Réconciliation de ces données afin de calculer le chiffre par produit et le chiffre d’affaires total réalisé.
Identification des vins premium avec utilisation des méthodes statistiques tel que le z-score et l’intervalle interquartile.
Ton rôle en tant que Data Engineer sera d’automatiser cette chaîne de traitement et d’analyse de données. 

Les responsables de données doivent recevoir les rapports des chiffres d’affaires tous les mois ainsi que les extractions des fichiers contenant les vins premium et les vins ordinaires. “

  

Fidèle à son habitude lors de l'arrivée d’un nouveau collaborateur, Laurent vous accompagne à votre poste de travail.

 

Il va vous faire suivre:

les données brutes par mail ; 
la méthode utilisée par Stéphane pour identifier les vins premium.
 

Vous recevez le mail de Laurent:

 

De : Laurent

À : moi

Objet : Exports tables et démarche de Stéphane

Re,

 

Tu trouveras ci-joints dans le fichier zip, les 2 exports du système ERP (erp.xlsx) et de la plateforme de vente en ligne CMS (web.xlsx) et le fichier liaison. Pour réconcilier les données, il faut passer par le fichier liaison (liaison.xlsx). 

 

Je te laisse prendre connaissance de ces éléments.

 

Voici la démarche de Stéphane : 

Suppression des valeurs manquantes et dédoublonnage des fichiers afin d’obtenir des clés primaires uniques avant les jointures entre les données.
Jointure des données communes entre ces deux fichiers propres en passant par le fichier intermédiaire liaison.xlsx.
Calcul des chiffres d’affaires par bouteille de vins et du chiffre d'affaires total sur les données fusionnées. 
Identification des vins premium en appliquant la méthode de z-score sur les prix des vins.
 

Ton rôle est d’industrialiser ce processus pour que les responsables produits puissent réaliser efficacement leurs ciblages marketing.

 

Au niveau de l’outil d'automatisation, notre DSI s’est positionné sur Kestra qui semble plus simple à prendre en main.

 

Nous souhaiterions présenter cette automatisation pour le prochain COPIL. Sur le plan technique, le DSI pourrait donner ses appréciations. Pourrais-tu préparer une présentation pour expliquer à l’ensemble de l’auditoire ta démarche et nous faire une démonstration ? L’objectif est de vulgariser la partie technique d’automatisation. 

 

Voici les éléments qui nous intéressent :

L’architecture de l’automatisation avec les procédures de tests. C'est-à-dire une conceptualisation des enchaînements des tâches de data transformation réalisées par Stéphane (nettoyages, jointures, agrégations, extractions) avec à l’issue de chaque tâche, une tâche de test pour vérifier que le résultat est juste.

L'implémentation de cette architecture avec les tests sur Kestra.

Une extraction du rapport au format Excel avec : 
* les chiffres d'affaires par produit ;
* le chiffre d'affaires total. 

Une extraction : 
* des vins premium ;
* des vins ordinaires.

Une planification de l'exécution de l’ensemble du workflow tous les 15 du mois à 9h.

Une solution si éventuellement il y avait des dysfonctionnements dans l’automatisation (indisponibilité d’un service tiers comme DuckDB par exemple).
 

N’hésite pas à solliciter Stéphane (sur la partie analyse de données) ou moi-même si tu as des questions.


Laurent

P.J. : 

Exports.zip
 

## Etape 1 - Concevez l'architecture d'automatisation (Data Lineage)



```mermaid

flowchart TD;
  DT[("Database")]
  e1["Import des données<br/>- erp.xlsx<br/>- liaison.xlsx<br/>- web.xlsx"]
  d1@{ shape: lean-r, label: "Données exportées" }
  A["Nettoyage:<br/>- suppression doublons<br/>- colonnes vides<br/>- lignes vides<br/>- formatage (date, texte, entiers...)"]
  d2@{ shape: lean-r, label: "Données nettoyées" }
  T1["Tests:<br/>- format de données,<br/>- unicité,<br/>- colonnes ou lignes vides"]
  c1{"Tests OK?"}
  V1["Vérifier les données"]
  B["Jointure:<br/>Jonction des tables erp et web<br/>via la table liaison"]
  d3@{ shape: lean-r, label: "Table Fusion" }
  T2["Tests:<br/>comparaison du nombre de lignes avant et après jonction"]
  c2{"Tests OK?"}
  V2["Vérifier les données"]
  C["Calcul du chiffre d'affaires (CA):<br/>- par vin,<br/>- total"]
  T3["Test de cohérence du CA:<br/>sommes des CA par vin vs CA total"]
  c3{"CA cohérent?"}
  V3["Vérifier les données"]
  D["Calcul du z-score sur le prix des vins"]
  E["Extraction rapport en Excel"]
  s1@{shape: doc, label: "Extrait CA par produit et total<br/>(fichier Excel)"}
  T4["Test de cohérence:<br/>prix vs seuil vin premium/vin ordinaire"]
  c4{"Test OK?"}
  V4["Vérifier les prix incohérents"]
  F["Séparation des données vins premium/ordinaires"]
  G-1["Extraction des données"]
  G-2["Extraction des données"]
  d4@{ shape: lean-r, label: "données vins premium" }
  d5@{ shape: lean-r, label: "données vins secondaires" }
  s2@{shape: doc, label: "Extrait vins premium<br/>(fichier CSV)"}
  s3@{shape: doc, label: "Extrait vins secondaires<br/>(fichier CSV)"}
  fin@{shape: terminal, label: "fin"}

  DT-->e1;
  e1-->d1;
  d1-->A;
  A-->d2;
  d2-->T1;
  T1-->c1;
  c1-->|"non"|V1;
  c1-->|"oui"| B;  
  V1-->A;  
  B-->d3;
  d3-->T2;
  T2-->c2;
  c2-->|"non"| V2;
  c2-->|"oui"| C;
  V2-->B;
  C-->T3;
  T3-->c3;
  c3-->|"non"| V3;
  c3-->|"oui"| D;
  c3--"oui"--> E;
  V3-->C;
  E-->s1;
  D-->T4;
  T4-->c4;
  c4-->|"non"| V4;
  c4-->|"oui"| F;
  V4-->D;
  F-->d4;
  F-->d5;
  d4-->G-1;
  d5-->G-2;
  G-1-->s2;
  G-2-->s3;
  s2-->fin;
  s3-->fin;
  
  style T1 fill:#fdebd0,stroke:#e67e22
  style T2 fill:#fdebd0,stroke:#e67e22
  style T3 fill:#fdebd0,stroke:#e67e22
  style T4 fill:#fdebd0,stroke:#e67e22
  style e1 fill:#d6eaf8,stroke:#2980b9
  style A fill:#d6eaf8,stroke:#2980b9  
  style B fill:#d6eaf8,stroke:#2980b9
  style C fill:#d6eaf8,stroke:#2980b9
  style D fill:#d6eaf8,stroke:#2980b9  
  style E fill:#d6eaf8,stroke:#2980b9
  style F fill:#d6eaf8,stroke:#2980b9
  style G-1 fill:#d6eaf8,stroke:#2980b9
  style G-2 fill:#d6eaf8,stroke:#2980b9
  style d1 fill:#d5f5e3,stroke:#27ae60
  style d2 fill:#d5f5e3,stroke:#27ae60
  style d3 fill:#d5f5e3,stroke:#27ae60
  style d4 fill:#d5f5e3,stroke:#27ae60
  style d5 fill:#d5f5e3,stroke:#27ae60
  style c1 fill:#ffdfe5,stroke:#ff5978
  style c2 fill:#ffdfe5,stroke:#ff5978
  style c3 fill:#ffdfe5,stroke:#ff5978
  style c4 fill:#ffdfe5,stroke:#ff5978
```

In [1]:
from IPython.display import IFrame

IFrame(src="Diagramme.drawio.html", width="100%", height="400")

Le pipeline démarre par l'import de trois fichiers sources depuis la base de données : erp.xlsx, liaison.xlsx et web.xlsx. Ces données brutes exportées passent d'abord par une étape de nettoyage, qui élimine les doublons, les colonnes et lignes vides, et harmonise le formatage (dates, chaînes de caractères, entiers). Un premier bloc de tests vérifie ensuite le format, l'unicité et l'absence de champs vides. Si un problème est détecté, les données repartent en arrière pour être corrigées ; sinon, le flux continue.

Vient ensuite la jointure des tables erp et web via la table de liaison, pour reconstituer une table fusionnée complète. Un deuxième contrôle compare le nombre de lignes avant et après la jonction, afin de s'assurer qu'aucune donnée n'a été perdue ou dupliquée par erreur.

Une fois les données validées, le pipeline calcule le chiffre d'affaires (par vin et au total), puis vérifie sa cohérence en comparant la somme des CA individuels au CA total. Ce chiffre d'affaires validé est alors exporté dans un rapport Excel.

En parallèle, un z-score est calculé sur les prix des vins pour distinguer statistiquement les vins premium des vins ordinaires. Un dernier test vérifie que cette classification respecte les seuils attendus. Les prix jugés incohérents sont réexaminés avant de poursuivre.

Enfin, les données sont séparées en deux groupes — vins premium et vins ordinaires — puis extraites séparément dans deux fichiers CSV distincts, marquant la fin du traitement.

## Etape 2 - Orchestrez les tâches nominales avec Kestra


### Installation de Kestra

Télécharger le docker-compose.yml nécessaire pour l'installation en tapant la commande suivante:
```
curl -o docker-compose.yml https://raw.githubusercontent.com/kestra-io/kestra/refs/heads/develop/docker-compose.yml
```
Le lancement de l'installation se fait via Docker avec la commande `docker compose up -d`

![installation_kestra.png](installation_kestra.png)

Une fois l'installation terminée, sur rendre dans un navigateur internet et taper l'URL http://localhost:8080/. La page ci-dessous apparaît pour la création du compte user admin:
![page_acceuil_kestra.png](page_acceuil_kestra.png)

Compléter le questionnaire:
![page_acceuil_kestra_questionnaire.png](page_acceuil_kestra_questionnaire.png)

Puis lancer l'interface UI Kestra:
![page_acceuil_kestra_start.png](page_acceuil_kestra_start.png)

L'interface se présente ainsi:
![page_acceuil_kestra_UI.png](page_acceuil_kestra_UI.png)

```
id: bottleneck_duckdb
namespace: bottleneck.team

tasks:
  - id: download_erp
    type: io.kestra.plugin.core.http.Download
    uri: "https://github.com/Sebules/Project_10_OpenClassrooms_Data_Engineer/raw/refs/heads/main/bottleneck/bottleneck/Fichier_erp.xlsx"

  - id: download_liaison
    type: io.kestra.plugin.core.http.Download
    uri: "https://github.com/Sebules/Project_10_OpenClassrooms_Data_Engineer/raw/refs/heads/main/bottleneck/bottleneck/fichier_liaison.xlsx"

  - id: download_web
    type: io.kestra.plugin.core.http.Download
    uri: "https://github.com/Sebules/Project_10_OpenClassrooms_Data_Engineer/raw/refs/heads/main/bottleneck/bottleneck/Fichier_web.xlsx"

  - id: creation_tables_duckdb
    type: io.kestra.plugin.scripts.python.Script
    
    containerImage: python:3.13-slim
    beforeCommands:
      - pip install --no-cache-dir duckdb kestra
    dependencyCacheEnabled: true

    inputFiles:
      Fichier_erp.xlsx: "{{ outputs.download_erp.uri }}"
      fichier_liaison.xlsx: "{{ outputs.download_liaison.uri }}"
      Fichier_web.xlsx: "{{ outputs.download_web.uri }}"
    outputFiles:
      - 'bottleneck.db'
    script: |
      import duckdb
      from pathlib import Path
      from kestra import Kestra

      conn = duckdb.connect()

      def creation_table(url,database:str):
        """
        Creation de tables dans la database DuckDB à partir d'un chemin d'accès. 
        Les fichiers sont des .xlsx. La fonction ne fonctionne que pour des fichiers Excel.
        """
        conn.sql(f"ATTACH IF NOT EXISTS '{database}.db' AS db;")
        tables_export = []
        for file in url:
            nom_table = Path(file).stem.split('_')[-1] + "_export_data"
            conn.sql(f"CREATE OR REPLACE TABLE db.{nom_table} AS SELECT * FROM read_xlsx('{file}',all_varchar=true );")
            print(f"table {nom_table} créée dans la database DuckDB {database}.db")
            print(conn.sql(f"FROM db.{nom_table} LIMIT 5"))
            tables_export.append(nom_table)            
        return tables_export
      
      fichiers = ['{{ workingDir }}/Fichier_erp.xlsx', '{{ workingDir }}/fichier_liaison.xlsx', '{{ workingDir }}/Fichier_web.xlsx']

      tables_export = creation_table(fichiers,'bottleneck')
      # On expose la valeur en tant qu'output Kestra
      Kestra.outputs({"tables_creees": tables_export})

      conn.close()

  - id: verif_table_creation
    type: io.kestra.plugin.jdbc.duckdb.Queries
    url: "jdbc:duckdb:"        
    databaseUri: "{{ outputs.creation_tables_duckdb.outputFiles['bottleneck.db'] }}"
    fetchType: STORE
    sql: |-
      ATTACH 'bottleneck.db';  
      SELECT * FROM 'erp_export_data' LIMIT 5;
      SELECT * FROM 'web_export_data' LIMIT 5;
      SELECT * FROM 'liaison_export_data' LIMIT 5;
    
  - id: suppression_cols_vide
    type: io.kestra.plugin.scripts.python.Script
    
    containerImage: python:3.13-slim
    beforeCommands:
      - pip install --no-cache-dir duckdb kestra
    dependencyCacheEnabled: true
    inputFiles: 
      bottleneck.db: "{{ outputs.creation_tables_duckdb.outputFiles['bottleneck.db']}}"     
    outputFiles:
      - 'bottleneck.db'
    script: |
      import duckdb
      from kestra import Kestra
      
      conn = duckdb.connect()
      tables_export = {{ outputs.creation_tables_duckdb.vars.tables_creees }}

      def delete_columns_null(tables, database):
          delete_columns_table=[]
          conn.sql(f"ATTACH IF NOT EXISTS '{database}.db';")
          # view the columns to be delete
          for table in tables:
              conn.sql(f"CREATE OR REPLACE VIEW {database}.{table}_summarized_view AS (SUMMARIZE {database}.{table});")
              print(f"vue {database}.{table} créée")
              print(conn.sql(f"SELECT * FROM {database}.{table}_summarized_view;"))

              null_columns=conn.sql(f"SELECT column_name FROM {database}.{table}_summarized_view WHERE null_percentage = 100.0;")

              if null_columns:
                  print(f"null columns from table {table} are\n{null_columns}")
                  list_null_columns = [row[0] for row in null_columns.fetchall()]
                  delete_columns_table.append((table,"null columns",list_null_columns[:]))
              else:
                  print(f"no null columns in table {table}")

              zeros_columns = conn.sql(f"SELECT column_name FROM {database}.{table}_summarized_view WHERE (min = '0.0' AND max = '0.0') OR (min = '0' AND max = '0') ;")
              
              if zeros_columns:
                  print(f"zeros columns from table {table} are\n{zeros_columns}")
                  list_zeros_columns = [row[0] for row in zeros_columns.fetchall()]
                  delete_columns_table.append((table,"zeros columns",list_zeros_columns[:]))
              else:
                  print(f"no zeros columns in table {table}")

          # create cleaned tables
          for table, _, columns in delete_columns_table:
              tables_cleaned_names = []
              columns_str = ", ".join(columns)
              conn.sql(f"CREATE OR REPLACE TABLE {database}.{table}_clean AS SELECT * EXCLUDE({columns_str}) FROM {database}.{table};")
              tables_cleaned_names.append(f"{table}_clean")

          print("COLUMNS BEFORE DELETING \n")
          print(conn.sql(f"DESCRIBE {database}.{table}"))
          print("COLUMNS AFTER DELETING \n")
          print(conn.sql(f"DESCRIBE {database}.{table}_clean"))
              
          return delete_columns_table, tables_cleaned_names

      delete_columns_table, tables_cleaned_names = delete_columns_null(tables_export,'bottleneck')
          
      Kestra.outputs({"tables_clean_crees": tables_cleaned_names, "colonnes_supprimees":delete_columns_table})

      conn.close()
          
  - id: verif_delete_null
    type: io.kestra.plugin.jdbc.duckdb.Queries
    url: "jdbc:duckdb:"        
    databaseUri: "{{ outputs.suppression_cols_vide.outputFiles['bottleneck.db'] }}"
    fetchType: STORE
    sql: |-
      ATTACH 'bottleneck.db'; 
      SELECT * FROM 'web_export_data_clean' LIMIT 5;
  
  - id: suppression_lignes_vides
    type: io.kestra.plugin.scripts.python.Script
    containerImage: python:3.13-slim
    beforeCommands:
      - pip install --no-cache-dir duckdb kestra
    dependencyCacheEnabled: true
    inputFiles: 
      bottleneck.db: "{{ outputs.suppression_cols_vide.outputFiles['bottleneck.db']}}"     
    outputFiles:
      - 'bottleneck.db'
    script: |
      import duckdb
      from kestra import Kestra
      
      conn = duckdb.connect()
      tables_export = {{ outputs.creation_tables_duckdb.vars.tables_creees }}
      tables_clean = {{ outputs.suppression_cols_vide.vars.tables_clean_crees }}

      def delete_null_row(tables,tables_cleaned,database):
        conn.sql(f"ATTACH IF NOT EXISTS '{database}.db';")
        for table in tables:
            # Create table_clean if not exists.
            if f"{table}_clean" not in tables_cleaned:
                conn.sql(f"""CREATE OR REPLACE TABLE {database}.{table}_clean
                    AS SELECT * FROM {database}.{table};
                    """)
                tables_cleaned.append(f"{table}_clean")
                print(f"table {table}_clean créée")

            
            nb_rows_before =conn.sql(f"SELECT COUNT(*) FROM {database}.{table}_clean;").fetchone()[0]
            print(conn.sql(f"CREATE OR REPLACE TABLE {database}.{table}_clean AS FROM {database}.{table}_clean EXCEPT FROM {database}.{table}_clean WHERE COLUMNS(*) IS NULL;"))
            nb_rows_after = conn.sql(f"SELECT COUNT(*) FROM {database}.{table}_clean;").fetchone()[0]
            print(nb_rows_before-nb_rows_after, f"null row(s) deleted in table {table}_clean")
                                
        return tables_cleaned
      tables_cleaned = delete_null_row(tables_export,tables_clean,'bottleneck')
      Kestra.outputs({"tables_clean_crees":tables_cleaned})

  - id: verif_delete_row
    type: io.kestra.plugin.jdbc.duckdb.Queries
    url: "jdbc:duckdb:"        
    databaseUri: "{{ outputs.suppression_lignes_vides.outputFiles['bottleneck.db'] }}"
    fetchType: STORE
    sql: |-
      ATTACH 'bottleneck.db'; 
      SELECT COUNT(*) FROM 'web_export_data_clean';
      SELECT COUNT(*) FROM 'web_export_data';
      SELECT COUNT(*) FROM 'erp_export_data_clean';
      SELECT COUNT(*) FROM 'erp_export_data';
      SELECT COUNT(*) FROM 'liaison_export_data_clean';
      SELECT COUNT(*) FROM 'liaison_export_data';
```


### Installation de DuckDB

Choix d'installation Duckdb via Python en utilisant la commande `pip install duckdb`

In [2]:
import duckdb
import os
from pathlib import Path
import pandas as pd

In [3]:
conn = duckdb.connect()

In [4]:
conn

In [5]:
conn.sql("SELECT 'world' as world;")

┌─────────┐
│  world  │
│ varchar │
├─────────┤
│ world   │
└─────────┘

Installation du CLI DuckDB en utilisant dans PowerShell la commande `winget install DuckDB.cli`

Lancement du CLI DuckDB:
```
(base) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_10>duckdb
DuckDB v1.5.5 (Variegata)
Enter ".help" for usage hints.
memory D 
```


### Lecture des fichiers export dans DuckDB

In [6]:
os.getcwd()

'C:\\Users\\sebas\\Documents\\cours_openclassrooms\\data_engineer\\projet_10'

In [7]:
current_path = Path(os.getcwd())

In [8]:
current_path

WindowsPath('C:/Users/sebas/Documents/cours_openclassrooms/data_engineer/projet_10')

In [9]:
os.listdir()

['.$Diagramme.drawio.bkp',
 '.git',
 '.gitignore',
 '.ipynb_checkpoints',
 'bottleneck',
 'bottleneck.db',
 'Diagramme.drawio',
 'Diagramme.drawio.html',
 'docker-compose.yml',
 'DuckDB_lecturefichier_creation_db.png',
 'DuckDB_lecturefichier_web.png',
 'installation_kestra.png',
 'notebook_projet_10.ipynb',
 'page_acceuil_kestra.png',
 'page_acceuil_kestra_questionnaire.png',
 'page_acceuil_kestra_start.png',
 'page_acceuil_kestra_UI.png',
 'showalltables.png',
 'SUMMARIZE_erp.png',
 'SUMMARIZE_liaison.png',
 'SUMMARIZE_web.png',
 '{database}.db']

In [10]:
file = os.listdir('bottleneck/bottleneck')

In [11]:
files_url = []
for f in file:
    url = os.path.join(current_path, 'bottleneck', 'bottleneck', f)
    url = url.replace('\\','/')
    files_url.append(url)

In [12]:
print(files_url)

['C:/Users/sebas/Documents/cours_openclassrooms/data_engineer/projet_10/bottleneck/bottleneck/Fichier_erp.xlsx', 'C:/Users/sebas/Documents/cours_openclassrooms/data_engineer/projet_10/bottleneck/bottleneck/fichier_liaison.xlsx', 'C:/Users/sebas/Documents/cours_openclassrooms/data_engineer/projet_10/bottleneck/bottleneck/Fichier_web.xlsx']


In [13]:
# lecture des fichiers d'export
def lecture_files(url):
    for file in url:
        nom_fichier = Path(file).stem
        tbl=conn.sql(f"FROM read_xlsx('{file}',all_varchar=true) LIMIT 5;")
        print(nom_fichier)
        print(tbl)

In [14]:
lecture_files(files_url)

Fichier_erp
┌────────────┬────────────┬────────────────────┬────────────────┬──────────────┐
│ product_id │ onsale_web │       price        │ stock_quantity │ stock_status │
│  varchar   │  varchar   │      varchar       │    varchar     │   varchar    │
├────────────┼────────────┼────────────────────┼────────────────┼──────────────┤
│ 3847       │ 1          │ 24.2               │ 0              │ outofstock   │
│ 3849       │ 1          │ 34.299999999999997 │ 0              │ outofstock   │
│ 3850       │ 1          │ 20.8               │ 0              │ outofstock   │
│ 4032       │ 1          │ 14.1               │ 0              │ outofstock   │
│ 4039       │ 1          │ 46                 │ 0              │ outofstock   │
└────────────┴────────────┴────────────────────┴────────────────┴──────────────┘

fichier_liaison
┌────────────┬─────────┐
│ product_id │ id_web  │
│  varchar   │ varchar │
├────────────┼─────────┤
│ 3847       │ 15298   │
│ 3849       │ 15296   │
│ 3850     

### Création des tables dans DuckDB

In [15]:
# Création des tables dans la database
def creation_table(url,database:str):
    """
    Creation de tables dans la database DuckDB à partir d'un chemin d'accès en local. 
    Les fichiers sont des .xlsx. La fonction ne fonctionne que pour des fichiers Excel.
    """
    conn.sql(f"ATTACH IF NOT EXISTS '{database}.db';")
    tables_export = []
    for file in url:
        nom_table = Path(file).stem.split('_')[-1] + "_export_data"
        conn.sql(f"CREATE OR REPLACE TABLE {database}.{nom_table} AS SELECT * FROM read_xlsx('{file}',all_varchar=true );")
        print(f"table {nom_table} créée dans la database DuckDB {database}.db")
        print(conn.sql(f"FROM {database}.{nom_table} LIMIT 5"))
        tables_export.append(nom_table)
    return tables_export

In [16]:
tables_export=creation_table(files_url, 'bottleneck')

table erp_export_data créée dans la database DuckDB bottleneck.db
┌────────────┬────────────┬────────────────────┬────────────────┬──────────────┐
│ product_id │ onsale_web │       price        │ stock_quantity │ stock_status │
│  varchar   │  varchar   │      varchar       │    varchar     │   varchar    │
├────────────┼────────────┼────────────────────┼────────────────┼──────────────┤
│ 3847       │ 1          │ 24.2               │ 0              │ outofstock   │
│ 3849       │ 1          │ 34.299999999999997 │ 0              │ outofstock   │
│ 3850       │ 1          │ 20.8               │ 0              │ outofstock   │
│ 4032       │ 1          │ 14.1               │ 0              │ outofstock   │
│ 4039       │ 1          │ 46                 │ 0              │ outofstock   │
└────────────┴────────────┴────────────────────┴────────────────┴──────────────┘

table liaison_export_data créée dans la database DuckDB bottleneck.db
┌────────────┬─────────┐
│ product_id │ id_web  │
│  

In [17]:
tables_export

['erp_export_data', 'liaison_export_data', 'web_export_data']

### Suppression des colonnes nulles

In [18]:
# Suppression des colonnes entièrement nulles
def delete_columns_null(tables, database):
    delete_columns_table=[]
    conn.sql(f"ATTACH IF NOT EXISTS '{database}.db';")
    # view the columns to be delete
    for table in tables:
        conn.sql(f"CREATE OR REPLACE VIEW {database}.{table}_summarized_view AS (SUMMARIZE {database}.{table});")
        print(f"vue {database}.{table} créée")
        print(conn.sql(f"SELECT * FROM {database}.{table}_summarized_view;"))

        null_columns=conn.sql(f"SELECT column_name FROM {database}.{table}_summarized_view WHERE null_percentage = 100.0;")

        if null_columns:
            print(f"null columns from table {table} are\n{null_columns}")
            list_null_columns = [row[0] for row in null_columns.fetchall()]
            delete_columns_table.append((table,"null columns",list_null_columns[:]))
        else:
            print(f"no null columns in table {table}")

        zeros_columns = conn.sql(f"SELECT column_name FROM {database}.{table}_summarized_view WHERE (min = '0.0' AND max = '0.0') OR (min = '0' AND max = '0') ;")
        
        if zeros_columns:
            print(f"zeros columns from table {table} are\n{zeros_columns}")
            list_zeros_columns = [row[0] for row in zeros_columns.fetchall()]
            delete_columns_table.append((table,"zeros columns",list_zeros_columns[:]))
        else:
            print(f"no zeros columns in table {table}")

    # create cleaned tables
    for table, _, columns in delete_columns_table:
        tables_cleaned_names = []
        columns_str = ", ".join(columns)
        conn.sql(f"CREATE OR REPLACE TABLE {database}.{table}_clean AS SELECT * EXCLUDE({columns_str}) FROM {database}.{table};")
        tables_cleaned_names.append(f"{table}_clean")

    print("COLUMNS BEFORE DELETING \n")
    print(conn.sql(f"DESCRIBE {database}.{table}"))
    print("COLUMNS AFTER DELETING \n")
    print(conn.sql(f"DESCRIBE {database}.{table}_clean"))

        
    return delete_columns_table, tables_cleaned_names
        
    

In [19]:
_, tables_cleaned_names = delete_columns_null(tables_export,'bottleneck')

vue bottleneck.erp_export_data créée
┌────────────────┬─────────────┬─────────┬────────────┬───────────────┬───────┬───────┬───────┬───────┬───────┬───────┬─────────────────┐
│  column_name   │ column_type │   min   │    max     │ approx_unique │  avg  │  std  │  q25  │  q50  │  q75  │ count │ null_percentage │
│    varchar     │   varchar   │ varchar │  varchar   │     int64     │ int32 │ int32 │ int32 │ int32 │ int32 │ int64 │  decimal(9,2)   │
├────────────────┼─────────────┼─────────┼────────────┼───────────────┼───────┼───────┼───────┼───────┼───────┼───────┼─────────────────┤
│ product_id     │ VARCHAR     │ 3847    │ 7338       │           764 │  NULL │  NULL │  NULL │  NULL │  NULL │   825 │            0.00 │
│ onsale_web     │ VARCHAR     │ 0       │ 1          │             2 │  NULL │  NULL │  NULL │  NULL │  NULL │   825 │            0.00 │
│ price          │ VARCHAR     │ -1      │ 99         │           411 │  NULL │  NULL │  NULL │  NULL │  NULL │   825 │            0.00

In [20]:
tables_cleaned_names

['web_export_data_clean']

In [21]:
tables_export

['erp_export_data', 'liaison_export_data', 'web_export_data']

### Suppression des produits avec un prix négatif

In [22]:
# Suppression et mise à part des produits avec un prix négatifs
def delete_negative_price(tables,tables_clean, database):
    conn.sql(f"ATTACH IF NOT EXISTS '{database}.db';")
    tables_negative_prices = []
    for table in tables:
        describe = conn.sql(f"SELECT column_name FROM (DESCRIBE {database}.{table});")
        describe_columns = [row[0] for row in describe.fetchall()]
        # Create table_clean if not exists.
        if f"{table}_clean" not in tables_clean:
            conn.sql(f"""CREATE OR REPLACE TABLE {database}.{table}_clean
                    AS SELECT * FROM {database}.{table};
                    """)
            tables_clean.append(f"{table}_clean")
            print(f"table {table}_clean créée")
        
        if 'price' in describe_columns:
            conn.sql(f"DELETE FROM {database}.{table}_clean WHERE TRY_CAST(price AS DOUBLE) <=0;")
            conn.sql(f"CREATE OR REPLACE TABLE {database}.{table}_negative_prices AS SELECT * FROM {database}.{table} WHERE TRY_CAST(price AS DOUBLE) <=0;")
            print(conn.sql(f"SELECT * FROM {database}.{table}_negative_prices LIMIT 5"))
            tables_negative_prices.append(f"{table}_negative_prices")
            print(conn.sql(f"SELECT * FROM (SUMMARIZE {database}.{table}_clean) WHERE column_name = 'price'" ))
            
        else:
            print(f"no price column in table {table}")
    
    return tables_clean, tables_negative_prices


In [23]:
tables_clean,tables_negative_prices = delete_negative_price(tables_export,tables_cleaned_names,'bottleneck')

table erp_export_data_clean créée
┌────────────┬────────────┬─────────┬────────────────┬──────────────┐
│ product_id │ onsale_web │  price  │ stock_quantity │ stock_status │
│  varchar   │  varchar   │ varchar │    varchar     │   varchar    │
├────────────┼────────────┼─────────┼────────────────┼──────────────┤
│ 5017       │ 0          │ -8      │ 0              │ outofstock   │
│ 6594       │ 0          │ -1      │ 192            │ instock      │
└────────────┴────────────┴─────────┴────────────────┴──────────────┘

┌─────────────┬─────────────┬─────────┬─────────┬───────────────┬───────┬───────┬───────┬───────┬───────┬───────┬─────────────────┐
│ column_name │ column_type │   min   │   max   │ approx_unique │  avg  │  std  │  q25  │  q50  │  q75  │ count │ null_percentage │
│   varchar   │   varchar   │ varchar │ varchar │     int64     │ int32 │ int32 │ int32 │ int32 │ int32 │ int64 │  decimal(9,2)   │
├─────────────┼─────────────┼─────────┼─────────┼───────────────┼───────┼──────

In [24]:
tables_clean

['web_export_data_clean', 'erp_export_data_clean', 'liaison_export_data_clean']

### Suppression des lignes nulles

In [25]:
# Suppression lignes entièrement nulles

def delete_null_row(tables,tables_clean,database):
    conn.sql(f"ATTACH IF NOT EXISTS '{database}.db';")
    for table in tables:
        # Create table_clean if not exists.
        if f"{table}_clean" not in tables_clean:
            conn.sql(f"""CREATE OR REPLACE TABLE {database}.{table}_clean
                 AS SELECT * FROM {database}.{table};
                 """)
            tables_clean.append(f"{table}_clean")
            print(f"table {table}_clean créée")

        
        nb_rows_before =conn.sql(f"SELECT COUNT(*) FROM {database}.{table}_clean;").fetchone()[0]
        conn.sql(f"""
                    CREATE OR REPLACE TABLE {database}.{table}_clean 
                    AS FROM {database}.{table}_clean 
                    EXCEPT FROM {database}.{table}_clean 
                    WHERE COLUMNS(*) IS NULL;
                    """)
        
        nb_rows_after = conn.sql(f"SELECT COUNT(*) FROM {database}.{table}_clean;").fetchone()[0]

        print( nb_rows_before ,f"rows before in table {table}_clean" )
        print( nb_rows_after ,f"rows after in table {table}_clean" )
        print(nb_rows_before-nb_rows_after, f"null row(s) deleted in table {table}_clean")
                            
    return tables_clean
    



In [26]:
tables_clean = delete_null_row(tables_export,tables_clean,'bottleneck')

823 rows before in table erp_export_data_clean
823 rows after in table erp_export_data_clean
0 null row(s) deleted in table erp_export_data_clean
825 rows before in table liaison_export_data_clean
825 rows after in table liaison_export_data_clean
0 null row(s) deleted in table liaison_export_data_clean
1513 rows before in table web_export_data_clean
1430 rows after in table web_export_data_clean
83 null row(s) deleted in table web_export_data_clean


In [27]:
tables_clean

['web_export_data_clean', 'erp_export_data_clean', 'liaison_export_data_clean']

In [28]:
#for table in tables_clean:
    #conn.sql(f"DELETE FROM 'bottleneck.{table}';")
    #print(table, " deleted")

### Dédoublement

In [29]:
# Dédoublement
def dedoublement(tables, tables_clean, database):
    conn.sql(f"ATTACH IF NOT EXISTS '{database}.db';")
    for table in tables:
        # Create table_clean if not exists.
        if f"{table}_clean" not in tables_clean:
            conn.sql(f"""CREATE OR REPLACE TABLE {database}.{table}_clean
                    AS SELECT * FROM {database}.{table};
                    """)
            tables_clean.append(f"{table}_clean")
            print(f"table {table}_clean créée")

        nb_rows_before =conn.sql(f"""
                                 SELECT COUNT(*) 
                                 FROM {database}.{table}_clean;
                                 """).fetchone()[0]
        conn.sql(f""" 
            CREATE OR REPLACE TABLE {database}.{table}_clean 
            AS (SELECT DISTINCT * FROM {database}.{table}_clean)
            """)
        nb_rows_after =conn.sql(f"""
                                 SELECT COUNT(*) 
                                 FROM {database}.{table}_clean;
                                 """).fetchone()[0]
        print( nb_rows_before ,f"rows before in table {table}_clean" )
        print( nb_rows_after ,f"rows after in table {table}_clean" )
        print(nb_rows_before-nb_rows_after, f"row(s) deleted in table {table}_clean")

    return tables_clean



In [30]:
tables_clean = dedoublement(tables_export, tables_clean, 'bottleneck')

823 rows before in table erp_export_data_clean
823 rows after in table erp_export_data_clean
0 row(s) deleted in table erp_export_data_clean
825 rows before in table liaison_export_data_clean
825 rows after in table liaison_export_data_clean
0 row(s) deleted in table liaison_export_data_clean
1430 rows before in table web_export_data_clean
1430 rows after in table web_export_data_clean
0 row(s) deleted in table web_export_data_clean


### Conversion format des données

In [31]:
# Conversion de format
def convert_format(tables, tables_clean, database):
    conn.sql(f"ATTACH IF NOT EXISTS '{database}.db';")
    list_conv_date = ['date', 'post_modified']
    list_conv_num = ['price', 'quantity', 'total']

    for table in tables:
        describe = conn.sql(f"SELECT column_name FROM (DESCRIBE {database}.{table});")
        describe_columns = [row[0] for row in describe.fetchall()]

        # Create table_clean if not exists.
        if f"{table}_clean" not in tables_clean:
            conn.sql(f"""CREATE OR REPLACE TABLE {database}.{table}_clean
                 AS SELECT * FROM {database}.{table};
                 """)
            tables_clean.append(f"{table}_clean")
            print(f"table {table}_clean créée")
            

        for col in describe_columns:
            col_lower = col.lower()

            # DATE conversion
            if any(word in col_lower for word in list_conv_date):
                conn.sql(f"""
                    ALTER TABLE {database}.{table}_clean 
                    ALTER COLUMN {col} TYPE DATE 
                    USING (DATE '1899-12-30' + CAST(TRY_CAST({col} AS DOUBLE) AS INTEGER) * INTERVAL 1 DAY);
                """)
                print(f"colonne {col} de {table}_clean convertie en DATE")

            # DOUBLE conversion
            elif any(word in col_lower for word in list_conv_num):
                conn.sql(f"""
                    ALTER TABLE {database}.{table}_clean 
                    ALTER COLUMN {col} TYPE DOUBLE 
                    USING TRY_CAST({col} AS DOUBLE);
                """)
                print(f"colonne {col} de {table}_clean convertie en DOUBLE")




    return tables_clean
                





In [32]:
convert_format(tables_export,tables_clean,'bottleneck')

colonne price de erp_export_data_clean convertie en DOUBLE
colonne stock_quantity de erp_export_data_clean convertie en DOUBLE
colonne total_sales de web_export_data_clean convertie en DOUBLE
colonne post_date de web_export_data_clean convertie en DATE
colonne post_date_gmt de web_export_data_clean convertie en DATE
colonne post_modified de web_export_data_clean convertie en DATE
colonne post_modified_gmt de web_export_data_clean convertie en DATE


['web_export_data_clean', 'erp_export_data_clean', 'liaison_export_data_clean']

In [33]:
conn.sql("FROM bottleneck.web_export_data_clean LIMIT 5")

┌─────────┬─────────────┬────────────┬───────────┬─────────────┬────────────┬───────────────┬──────────────┬─────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────┬────────────────┬─────────────┬───────────────┬─────────────────────────────────────────────────────┬───────────────┬───────────────────┬───────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────┬────────────────┐
│   sku   │ total_sales │ tax_status │ tax_class │ post_author │ post_date  │ post_date_gmt │ post_content │                       post_title                        │                                                                                    post_excerpt                                                                         

In [34]:
conn.sql("FROM bottleneck.web_export_data LIMIT 5")

┌─────────┬─────────┬──────────────┬──────────────┬────────────────┬─────────────┬────────────┬───────────┬─────────────┬────────────────────┬────────────────────┬──────────────┬──────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────┬────────────────┬─────────────┬───────────────┬──────────────────────────────────────────────────────────────┬────────────────────┬────────────────────┬───────────────────────┬─────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────┬────────────┬────────────────┬────────────

### Suppression des lignes bons cadeau

In [35]:
# Suppression des lignes de la table web_export_data_clean qui correspondent à des bons cadeau

print("nombre de lignes avant suppression: \n" )
print(conn.sql(f"SELECT COUNT(*) FROM bottleneck.web_export_data_clean;"))
conn.sql(f"""
    DELETE FROM bottleneck.web_export_data_clean
    WHERE sku = 'bon-cadeau-25-euros';
    """)
print("nombre de lignes après suppression:\n" )
print(conn.sql(f"SELECT COUNT(*) FROM bottleneck.web_export_data_clean;"))

nombre de lignes avant suppression: 

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1430 │
└──────────────┘

nombre de lignes après suppression:

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1428 │
└──────────────┘



### Suppression des lignes attachments

In [36]:
# Suppression des lignes la table web_export_data_clean qui correspondent à des attachments (image) et non des produits.

print("nombre de lignes avant suppression:\n" )
print(conn.sql(f"SELECT COUNT(*) FROM bottleneck.web_export_data_clean;"))
conn.sql(f"""
    DELETE FROM bottleneck.web_export_data_clean
    WHERE post_type = 'attachment';
    """)
print("nombre de lignes après suppression:\n" )
print(conn.sql(f"SELECT COUNT(*) FROM bottleneck.web_export_data_clean;"))

nombre de lignes avant suppression:

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1428 │
└──────────────┘

nombre de lignes après suppression:

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          715 │
└──────────────┘



### Calcul du Z-score et classification premium

In [37]:
# Z-score
conn.sql(f"ATTACH IF NOT EXISTS 'bottleneck.db';")
avg_std=conn.sql("""
    SELECT column_name, avg, std
    FROM (SUMMARIZE bottleneck.erp_export_data_clean)
    WHERE column_name = 'price'
    """).fetchall()



In [38]:
avg_std

[('price', '32.473633049817735', '26.802511453964154')]

In [42]:
avg_price = float(avg_std[0][1])
std_price = float(avg_std[0][2])

In [43]:
avg_price

32.473633049817735

In [44]:
std_price

26.802511453964154

In [77]:
#Script Python calcul Z-score
data_products = conn.sql("SELECT * FROM bottleneck.erp_export_data_clean").df()
data_products['z-score']=(
    (data_products['price']-data_products['price'].mean())
    /data_products['price'].std()
)
data_products['premium']=0
data_products.loc[data_products['z-score']>2,'premium']=1

# Chargement de la database dans DuckDB
conn.execute("""
            CREATE OR REPLACE TABLE bottleneck.erp_export_data_clean_z
            AS (SELECT *
            FROM data_products);
            """
            )
conn.sql("FROM bottleneck.erp_export_data_clean_z LIMIT 5")

┌────────────┬────────────┬────────┬────────────────┬──────────────┬──────────────────────┬─────────┐
│ product_id │ onsale_web │ price  │ stock_quantity │ stock_status │       z-score        │ premium │
│  varchar   │  varchar   │ double │     double     │   varchar    │        double        │  int64  │
├────────────┼────────────┼────────┼────────────────┼──────────────┼──────────────────────┼─────────┤
│ 4165       │ 1          │   12.0 │           57.0 │ instock      │  -0.7638699487166768 │       0 │
│ 4195       │ 0          │   14.1 │            0.0 │ outofstock   │  -0.6855190820970711 │       0 │
│ 4264       │ 1          │   17.8 │           48.0 │ instock      │  -0.5474723171006229 │       0 │
│ 4600       │ 1          │   26.5 │            0.0 │ outofstock   │ -0.22287586967654213 │       0 │
│ 4610       │ 1          │   13.1 │          114.0 │ instock      │  -0.7228290185825976 │       0 │
└────────────┴────────────┴────────┴────────────────┴──────────────┴──────────────

### Fusion des tables erp et web

In [88]:
# Join tables erp and web
conn.sql(f"ATTACH IF NOT EXISTS 'bottleneck.db';")
conn.sql("""
    CREATE OR REPLACE TABLE bottleneck.web_erp_joined AS (
    SELECT ec.*,
        wc.*,
    FROM bottleneck.web_export_data_clean wc
    JOIN bottleneck.liaison_export_data_clean lc ON wc.sku = lc.id_web
    JOIN bottleneck.erp_export_data_clean_z ec ON ec.product_id = lc.product_id
    );
    """)

In [89]:
# check number of rows
conn.sql(f"ATTACH IF NOT EXISTS 'bottleneck.db';")
conn.sql("SELECT COUNT(*) FROM bottleneck.web_erp_joined;")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          713 │
└──────────────┘

In [90]:
conn.sql(f"ATTACH IF NOT EXISTS 'bottleneck.db';")
conn.sql("SELECT DISTINCT COUNT(sku) FROM bottleneck.web_export_data_clean ")

┌────────────┐
│ count(sku) │
│   int64    │
├────────────┤
│        713 │
└────────────┘

In [93]:
conn.sql("SHOW ALL TABLES;").df()

,database,schema,name,column_names,column_types,temporary
0,bottleneck,main,erp_export_data,"[product_id, onsale_web, price, stock_quantity...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR]",False
1,bottleneck,main,erp_export_data_clean,"[product_id, onsale_web, price, stock_quantity...","[VARCHAR, VARCHAR, DOUBLE, DOUBLE, VARCHAR]",False
2,bottleneck,main,erp_export_data_clean_z,"[product_id, onsale_web, price, stock_quantity...","[VARCHAR, VARCHAR, DOUBLE, DOUBLE, VARCHAR, DO...",False
3,bottleneck,main,erp_export_data_negative_prices,"[product_id, onsale_web, price, stock_quantity...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR]",False
4,bottleneck,main,erp_export_data_summarized_view,"[column_name, column_type, min, max, approx_un...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, BIGINT, ""...",False
5,bottleneck,main,liaison_export_data,"[product_id, id_web]","[VARCHAR, VARCHAR]",False
6,bottleneck,main,liaison_export_data_clean,"[product_id, id_web]","[VARCHAR, VARCHAR]",False
7,bottleneck,main,liaison_export_data_summarized_view,"[column_name, column_type, min, max, approx_un...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, BIGINT, ""...",False
8,bottleneck,main,view_web_export_data_summary,"[column_name, column_type, min, max, approx_un...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, BIGINT, ""...",False
9,bottleneck,main,web_erp_joined,"[product_id, onsale_web, price, stock_quantity...","[VARCHAR, VARCHAR, DOUBLE, DOUBLE, VARCHAR, DO...",False


In [92]:
conn.sql("SUMMARIZE FROM bottleneck.web_erp_joined;").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,product_id,VARCHAR,3847,7338,680,None,None,None,None,None,713,0.0
1,onsale_web,VARCHAR,1,1,1,None,None,None,None,None,713,0.0
2,price,DOUBLE,5.2,225.0,371,32.50364656381488,27.82862912619748,14.028549382716049,23.5603305785124,42.36512345679012,713,0.0
3,stock_quantity,DOUBLE,-1.0,578.0,155,28.802244039270686,48.02707447469254,2.0,12.462809917355372,35.135802469135804,713,0.0
4,stock_status,VARCHAR,instock,outofstock,2,None,None,None,None,None,713,0.0
5,z-score,DOUBLE,-1.0175775168182573,7.183146522700475,334,0.001119802300940164,1.0382843851777015,-0.6881849000897868,-0.3325547490806891,0.36905087883133453,713,0.0
6,premium,BIGINT,0,1,2,0.043478260869565216,0.2040742797993488,0,0,0,713,0.0
7,sku,VARCHAR,10014,9937,693,None,None,None,None,None,713,0.0
8,total_sales,DOUBLE,0.0,96.0,45,4.004207573632539,8.521205156997404,0.0,1.0,4.135802469135802,713,0.0
9,tax_status,VARCHAR,taxable,taxable,1,None,None,None,None,None,713,0.0


In [94]:
conn.sql("SELECT * FROM bottleneck.web_erp_joined LIMIT 5;").df()

,product_id,onsale_web,price,stock_quantity,stock_status,z-score,premium,sku,total_sales,tax_status,...,comment_status,ping_status,post_password,post_name,post_modified,post_modified_gmt,post_content_filtered,guid,post_type,post_mime_type
0,4286,1,69.8,4.0,instock,1.392644,0,16066,10.0,taxable,...,closed,closed,None,hauvette-baux-provence-amethyste-2017,2020-07-17,2020-07-17,None,https://www.bottle-neck.fr/?post_type=product&...,product,None
1,5025,1,112.0,0.0,outofstock,2.967124,1,13914,0.0,taxable,...,closed,closed,None,champagne-agrapart-fils-lavizoise-grand-cru-20...,2020-07-10,2020-07-10,None,https://www.bottle-neck.fr/?post_type=product&...,product,None
2,4795,1,12.0,34.0,instock,-0.763870,0,3568,10.0,taxable,...,closed,closed,None,emile-boeckel-cremant-brut-rose,2020-08-25,2020-08-25,None,https://www.bottle-neck.fr/?post_type=product&...,product,None
3,6094,1,13.4,2.0,instock,-0.711636,0,15486,0.0,taxable,...,closed,closed,None,chateau-de-villeneuve-saumur-champigny-bienboi...,2020-05-13,2020-05-13,None,https://www.bottle-neck.fr/?post_type=product&...,product,None
4,4912,1,25.9,10.0,instock,-0.245262,0,15612,6.0,taxable,...,closed,closed,None,jean-baptiste-arena-patrimonio-rouge-grotte-di...,2020-08-08,2020-08-08,None,https://www.bottle-neck.fr/?post_type=product&...,product,None


### Calcul du chiffre d'affaires par produit et total

In [108]:
conn.sql("""
    CREATE OR REPLACE TABLE bottleneck.revenus_web AS
    SELECT 
        product_id,
        price,
        total_sales,
        price*total_sales AS CA_product
    FROM bottleneck.web_erp_joined;
    """)
conn.sql("FROM revenus_web;")

┌────────────┬────────┬─────────────┬────────────────────┐
│ product_id │ price  │ total_sales │     CA_product     │
│  varchar   │ double │   double    │       double       │
├────────────┼────────┼─────────────┼────────────────────┤
│ 4286       │   69.8 │        10.0 │              698.0 │
│ 5025       │  112.0 │         0.0 │                0.0 │
│ 4795       │   12.0 │        10.0 │              120.0 │
│ 6094       │   13.4 │         0.0 │                0.0 │
│ 4912       │   25.9 │         6.0 │ 155.39999999999998 │
│ 4607       │   13.4 │         0.0 │                0.0 │
│ 4675       │   10.7 │         0.0 │                0.0 │
│ 4933       │   18.4 │         0.0 │                0.0 │
│ 4680       │    6.3 │         2.0 │               12.6 │
│ 4718       │   18.2 │         0.0 │                0.0 │
│  ·         │     ·  │          ·  │                 ·  │
│  ·         │     ·  │          ·  │                 ·  │
│  ·         │     ·  │          ·  │                 · 

In [103]:
conn.sql("SELECT SUM(price*total_sales) AS CA_global FROM revenus_web;")

┌───────────────────┐
│     CA_global     │
│      double       │
├───────────────────┤
│ 70318.60000000003 │
└───────────────────┘

### Extraction de données en .xlsx et .csv

In [111]:
conn.sql("""
    COPY (
        SELECT * 
        FROM bottleneck.web_erp_joined 
        WHERE premium = 1
        )
    TO 'vins_premium.csv' (HEADER, DELIMITER ',');
    """)

conn.sql("""
    COPY (
        SELECT * 
        FROM bottleneck.web_erp_joined 
        WHERE premium = 0
        )
    TO 'vins_ordinaires.csv' (HEADER, DELIMITER ',');
    """)

conn.sql("""
    COPY (
        SELECT * 
        FROM bottleneck.revenus_web 
        )
    TO 'ca_products.xlsx'
    WITH (FORMAT xlsx, HEADER true, SHEET 'CA_products');
    """)

## Etape 3- Implémentez des tâches de test avec Kestra

In [112]:
conn.close()

### Lecture des fichiers exports dans DuckDB

avec les commandes suivantes dans le CLI:
```
memory D FROM 'C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_10\bottleneck\bottleneck\Fichier_erp.xlsx' LIMIT 5;
┌────────────┬────────────┬────────┬────────────────┬──────────────┐
│ product_id │ onsale_web │ price  │ stock_quantity │ stock_status │
│   double   │   double   │ double │     double     │   varchar    │
├────────────┼────────────┼────────┼────────────────┼──────────────┤
│     3847.0 │        1.0 │   24.2 │            0.0 │ outofstock   │
│     3849.0 │        1.0 │   34.3 │            0.0 │ outofstock   │
│     3850.0 │        1.0 │   20.8 │            0.0 │ outofstock   │
│     4032.0 │        1.0 │   14.1 │            0.0 │ outofstock   │
│     4039.0 │        1.0 │   46.0 │            0.0 │ outofstock   │
└────────────┴────────────┴────────┴────────────────┴──────────────┘
```

La lecture du fichier web.xlsx présente une erreur. 
```
memory D FROM 'C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_10\bottleneck\bottleneck\Fichier_web.xlsx' LIMIT 5;
Invalid Input Error:
read_xlsx: Failed to parse cell 'A198': Could not convert string 'bon-cadeau-25-euros' to DOUBLE
```

Utilisation de la commande suivante pour lire ce fichier:
```
memory D FROM read_xlsx('C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_10\bottleneck\bottleneck\Fichier_web.xlsx', all_varchar=true) LIMIT 5;
┌─────────┬─────────┬──────────────┬──────────────┬────────────────┬─────────────┬────────────┬───────────┬───┬───────────────────────┬─────────────┬────────────────────────────┬────────────┬────────────┬────────────────┬───────────────┐
│   sku   │ virtual │ downloadable │ rating_count │ average_rating │ total_sales │ tax_status │ tax_class │ … │ post_content_filtered │ post_parent │            guid            │ menu_order │ post_type  │ post_mime_type │ comment_count │
│ varchar │ varchar │   varchar    │   varchar    │    varchar     │   varchar   │  varchar   │  varchar  │ … │        varchar        │   varchar   │          varchar           │  varchar   │  varchar   │    varchar     │    varchar    │
├─────────┼─────────┼──────────────┼──────────────┼────────────────┼─────────────┼────────────┼───────────┼───┼───────────────────────┼─────────────┼────────────────────────────┼────────────┼────────────┼────────────────┼───────────────┤
│ 16004   │ 0       │ 0            │ 0            │ 0              │ 5           │ NULL       │ NULL      │ … │ NULL                  │ 0           │ https://www.bottle-neck.f… │ 0          │ attachment │ image/jpeg     │ 0             │
│ NULL    │ 0       │ 0            │ 0            │ NULL           │ NULL        │ NULL       │ NULL      │ … │ NULL                  │ NULL        │ NULL                       │ NULL       │ NULL       │ NULL           │ NULL          │
│ 15075   │ 0       │ 0            │ 0            │ 0              │ 3           │ taxable    │ NULL      │ … │ NULL                  │ 0           │ https://www.bottle-neck.f… │ 0          │ product    │ NULL           │ 0             │
│ 16209   │ 0       │ 0            │ 0            │ 0              │ 6           │ taxable    │ NULL      │ … │ NULL                  │ 0           │ https://www.bottle-neck.f… │ 0          │ product    │ NULL           │ 0             │
│ 15763   │ 0       │ 0            │ 0            │ 0              │ 1           │ NULL       │ NULL      │ … │ NULL                  │ 0           │ https://www.bottle-neck.f… │ 0          │ attachment │ image/jpeg     │ 0             │
└─────────┴─────────┴──────────────┴──────────────┴────────────────┴─────────────┴────────────┴───────────┴───┴───────────────────────┴─────────────┴────────────────────────────┴────────────┴────────────┴────────────────┴───────────────┘
  5 rows                                                                                       use .last to show entire result                                                                                        28 columns (15 shown)
memory D
```
![DuckDB_lecturefichier_web.png](DuckDB_lecturefichier_web.png)


Lecture du fichier liaison:
```
memory D FROM read_xlsx('C:/Users/sebas/Documents/cours_openclassrooms/data_engineer/projet_10/bottleneck/bottleneck/fichier_liaison.xlsx', all_varchar=true)
         LIMIT 5;
┌────────────┬─────────┐
│ product_id │ id_web  │
│  varchar   │ varchar │
├────────────┼─────────┤
│ 3847       │ 15298   │
│ 3849       │ 15296   │
│ 3850       │ 15300   │
│ 4032       │ 19814   │
│ 4039       │ 19815   │
└────────────┴─────────┘
memory D
```

Création de la database dans DuckDB avec la commande suivante:
```
memory D ATTACH 'bottleneck.db';

```

Création et visualisation de la table `erp_export_data`
```
memory D CREATE TABLE bottleneck.erp_export_data AS (SELECT * FROM 'C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_10\bottleneck\bottleneck\Fichier_erp.xlsx');
memory D FROM bottleneck.erp_export_data LIMIT 5;
┌────────────┬────────────┬────────┬────────────────┬──────────────┐
│ product_id │ onsale_web │ price  │ stock_quantity │ stock_status │
│   double   │   double   │ double │     double     │   varchar    │
├────────────┼────────────┼────────┼────────────────┼──────────────┤
│     3847.0 │        1.0 │   24.2 │            0.0 │ outofstock   │
│     3849.0 │        1.0 │   34.3 │            0.0 │ outofstock   │
│     3850.0 │        1.0 │   20.8 │            0.0 │ outofstock   │
│     4032.0 │        1.0 │   14.1 │            0.0 │ outofstock   │
│     4039.0 │        1.0 │   46.0 │            0.0 │ outofstock   │
└────────────┴────────────┴────────┴────────────────┴──────────────┘
```
![DuckDB_lecturefichier_creation_db.png](DuckDB_lecturefichier_creation_db.png)

Création des autres tables:

```
memory D CREATE TABLE bottleneck.web_export_data AS SELECT * FROM read_xlsx('C:/Users/sebas/Documents/cours_openclassrooms/data_engineer/projet_10/bottleneck/bottleneck/Fichier_web.xlsx', all_varchar=true);
memory D FROM botteneck.web_export_data LIMIT 5;
Catalog Error:
Table with name "botteneck.web_export_data" does not exist because schema "botteneck" does not exist.

LINE 1: FROM botteneck.web_export_data LIMIT 5;
             ^
memory D FROM bottleneck.web_export_data LIMIT 5;
```

```
memory D CREATE TABLE bottleneck.liaison_export_data AS SELECT * FROM read_xlsx('C:/Users/sebas/Documents/cours_openclassrooms/data_engineer/projet_10/bottleneck/bottleneck/fichier_liaison.xlsx', all_varchar=true);
memory D FROM bottleneck.liaison_export_data LIMIT 5;
┌────────────┬─────────┐
│ product_id │ id_web  │
│  varchar   │ varchar │
├────────────┼─────────┤
│ 3847       │ 15298   │
│ 3849       │ 15296   │
│ 3850       │ 15300   │
│ 4032       │ 19814   │
│ 4039       │ 19815   │
└────────────┴─────────┘
memory D
```
Vérifions la création de toutes les tables:
![showalltables.png](showalltables.png)



### Repérer les colonnes nulles ou complètement remplies de zéro

Avec la commande `SUMMARIZE` on peut identifier les colonnes nulles. 

**Pour l'export data Web:**
```
memory D CREATE VIEW bottleneck.view_web_export_data_summary AS SELECT * FROM (SUMMARIZE bottleneck.web_export_data);
```

La commande `SELECT * FROM bottleneck.view_web_export_data_summary;` permet de constater qu'il y a des colonnes qui sont entièrement nulles et d'autres qui ont uniquement des zéros.
![SUMMARIZE_web.png](SUMMARIZE_web.png)

```
memory D SELECT column_name FROM bottleneck.view_web_export_data_summary WHERE null_percentage = 100.00;
┌───────────────────────┐
│      column_name      │
│        varchar        │
├───────────────────────┤
│ tax_class             │
│ post_content          │
│ post_password         │
│ post_content_filtered │
└───────────────────────┘
memory D SELECT column_name FROM bottleneck.view_web_export_data_summary WHERE min='0' AND max='0';
┌────────────────┐
│  column_name   │
│    varchar     │
├────────────────┤
│ virtual        │
│ downloadable   │
│ rating_count   │
│ average_rating │
│ post_parent    │
│ menu_order     │
│ comment_count  │
└────────────────┘
memory D
```

Créer une nouvelle table `web_export_data_clean` qui ne contient pas ces colonnes:
```
memory D CREATE TABLE bottleneck.web_export_data_clean AS (
         SELECT * EXCLUDE ("virtual", downloadable, rating_count, average_rating, post_parent, menu_order, comment_count, tax_class, post_content, post_password, post_content_filtered) FROM bottleneck.web_export_data);

memory D DESCRIBE bottleneck.web_export_data_clean;
┌───────────────────────────┐
│   web_export_data_clean   │
│                           │
│ sku               varchar │
│ total_sales       varchar │
│ tax_status        varchar │
│ post_author       varchar │
│ post_date         varchar │
│ post_date_gmt     varchar │
│ post_title        varchar │
│ post_excerpt      varchar │
│ post_status       varchar │
│ comment_status    varchar │
│ ping_status       varchar │
│ post_name         varchar │
│ post_modified     varchar │
│ post_modified_gmt varchar │
│ guid              varchar │
│ post_type         varchar │
│ post_mime_type    varchar │
└───────────────────────────┘
memory D

```
Les colonnes exclues ne sont plus présentes.

**Pour l'export data erp:**

Le summarize indique qu'il n'y a pas de colonne 100% nulle ou remplie de zéros.
![SUMMARIZE_erp.png](SUMMARIZE_erp.png)

**Pour la table de liaison:**

Le summarize indique qu'il n'y a pas de colonne 100% nulle ou remplie de zéros.
![SUMMARIZE_liaison.png](SUMMARIZE_liaison.png)